# Efficient-SAM3-Finetuning — GPU Test Runner

Per-test cells for GPU-gated tests. Press play on the cell for the test you want;
the cleanup trailer runs after every test (pass or fail), so the next test starts
from a clean slate without a full runtime restart.

**Prereqs:**
1. Runtime → Change runtime type → **T4 GPU** (or better).
2. In Colab Secrets (left sidebar 🔑), add:
   - `HF_TOKEN` — Hugging Face access token with read access to gated `facebook/sam3.1`.
   - `GH_TOKEN` — GitHub fine-grained personal access token with **Contents: Read** on this repo. **Required only if the repo is private**; can be omitted for public repos.
   - `BRANCH` *(optional)* — feature branch to test (e.g. `manual-gpu-pass-44`). Used only if the notebook cannot detect the branch from the Colab URL.
3. Run setup cells (collapsed below) once per session.
4. Press play on each test cell in turn. Cells 10–14 are the canonical merge gate — every cell green = PR ready.


### GPU guard (runs first)

This first code cell asserts a CUDA GPU is available before any install
or test step. Colab can pin the **accelerator class** (GPU vs. CPU) via
notebook metadata (`metadata.colab.accelerator: "GPU"`), but it cannot
pin the **GPU model** — the most common assignment is a T4, but Colab
may serve a V100, A100, L4, or others depending on availability. The
test suite is calibrated for a T4; the guard warns (does not fail) if a
different GPU is detected.


In [ ]:
# GPU guard. Runs FIRST so a misconfigured runtime fails loudly before any
# slow install or test step. Two assertions:
#   1. nvidia-smi must succeed AND report at least one GPU. If it doesn't,
#      the runtime is CPU-only — change Runtime → Change runtime type → GPU.
#   2. If the GPU is not a T4, print a WARN: the test suite's timing /
#      memory assumptions are calibrated for a Colab T4, and other GPUs
#      (V100, A100, L4, …) may show different characteristics. The cell does
#      NOT fail in that case — other GPUs are usually fine, just unverified.
import subprocess
import sys

try:
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        check=True,
        capture_output=True,
        text=True,
    )
except FileNotFoundError as e:
    raise RuntimeError(
        "nvidia-smi not found; this runtime is CPU-only. "
        "Runtime → Change runtime type → GPU (T4 recommended) → Save."
    ) from e
except subprocess.CalledProcessError as e:
    raise RuntimeError(
        f"nvidia-smi failed (returncode={e.returncode}). "
        f"stderr: {e.stderr.strip()}"
    ) from e

gpus = [line.strip() for line in out.stdout.splitlines() if line.strip()]
if not gpus:
    raise RuntimeError(
        "nvidia-smi reported no GPUs. "
        "Runtime → Change runtime type → GPU (T4 recommended) → Save."
    )

print(f"GPU(s) detected: {gpus}")
if not any("T4" in g for g in gpus):
    print(
        f"WARN: detected GPU is not a T4 ({gpus}). The test suite's timing "
        "and memory assumptions are calibrated for a T4; other GPUs are "
        "usually fine but unverified.",
        file=sys.stderr,
    )


In [ ]:
# Cell 0: Config. Auto-detects the working branch so you don't edit the
# notebook on every test cycle. Resolution order (first non-empty wins):
#   1. BRANCH literal set below (manual override).
#   2. Colab notebook metadata — Colab populates metadata.colab.provenance
#      with the source URL when loaded from a Colab badge OR a direct
#      GitHub URL. Most reliable mechanism in practice.
#   3. URL parse from document.referrer — fragile (often empty under
#      Colab's referrer policy), kept as a fallback for the rare case
#      where (2) misses.
#   4. BRANCH Colab Secret (left sidebar key). One-time set, persists
#      across notebook sessions on the same Google account.
#   5. "main" — last-resort fallback. WARNs loudly with the exact URL
#      pattern to open this notebook from a specific PR branch.
import re
import urllib.parse

REPO = "NguyenJus/Efficient-SAM3-Finetuning"
NOTEBOOK_PATH = "notebooks/colab_gpu_tests.ipynb"
BRANCH = None  # set to a string literal to force a branch

# Match BOTH Colab badge URLs (colab.research.google.com/github/...) AND
# direct GitHub URLs (github.com/...). The previous regex required the
# `/github/` Colab-specific prefix, so GitHub-direct URLs in Colab's
# `provenance` metadata silently failed to match — the user-visible
# symptom was BRANCH falling back to "main" even when opened from a PR
# branch's GitHub view.
_BRANCH_RE = re.compile(
    r"(?:colab\.research\.google\.com/github/|github\.com/)[^/]+/[^/]+/blob/([^/?#]+)/",
    re.IGNORECASE,
)


def _from_ipynb_metadata() -> str | None:
    try:
        from google.colab import _message  # type: ignore

        meta = _message.blocking_request("get_ipynb", timeout_sec=5)["ipynb"]["metadata"]
    except Exception:
        return None
    for p in meta.get("colab", {}).get("provenance", []) or []:
        if not isinstance(p, dict):
            continue
        for v in p.values():
            if isinstance(v, str):
                m = _BRANCH_RE.search(v)
                if m:
                    return urllib.parse.unquote(m.group(1))
    return None


def _from_url() -> str | None:
    try:
        from google.colab import output  # type: ignore
    except Exception:
        return None
    try:
        url = output.eval_js(
            "(() => { try { return document.referrer || \"\"; } catch(e) { return \"\"; } })()",
            timeout_sec=5,
        )
    except Exception:
        return None
    if not url:
        return None
    m = _BRANCH_RE.search(url)
    return urllib.parse.unquote(m.group(1)) if m else None


def _from_secret() -> str | None:
    try:
        from google.colab import userdata  # type: ignore

        v = userdata.get("BRANCH")
    except Exception:
        return None
    return v or None


_diagnostics: dict[str, str] = {}
if BRANCH is None:
    for label, fn in (
        ("metadata", _from_ipynb_metadata),
        ("url", _from_url),
        ("secret", _from_secret),
    ):
        candidate = fn()
        _diagnostics[label] = candidate or "(none)"
        if candidate:
            BRANCH, _source = candidate, label
            break
    else:
        BRANCH, _source = "main", "fallback"
else:
    _source = "literal"

print(f"REPO   = {REPO}")
print(f"BRANCH = {BRANCH}  (source: {_source})")
if _source == "fallback":
    import sys

    print("", file=sys.stderr)
    print("WARN: BRANCH detection fell back to 'main'.", file=sys.stderr)
    print("Per-method results:", file=sys.stderr)
    for k, v in _diagnostics.items():
        print(f"  {k}: {v}", file=sys.stderr)
    print("", file=sys.stderr)
    print("To run this notebook against a specific PR branch, do ONE of:", file=sys.stderr)
    print(
        f"  a) Open https://colab.research.google.com/github/{REPO}/blob/<BRANCH>/{NOTEBOOK_PATH}",
        file=sys.stderr,
    )
    print("  b) Set the BRANCH Colab Secret (left sidebar key) to <BRANCH>.", file=sys.stderr)
    print('  c) Edit `BRANCH = "<BRANCH>"` in this cell.', file=sys.stderr)


In [ ]:
# Cell 1: Clone + checkout. Done BEFORE any torch/numpy import so the pip
# install in Cell 2 can downgrade numpy without Colab forcing a runtime restart.
import os
import subprocess

try:
    from google.colab import userdata  # type: ignore

    gh_token = userdata.get("GH_TOKEN")
except Exception:
    gh_token = None

clone_url = (
    f"https://{gh_token}@github.com/{REPO}.git" if gh_token else f"https://github.com/{REPO}.git"
)

if not os.path.isdir("Efficient-SAM3-Finetuning"):
    subprocess.run(["git", "clone", clone_url], check=True)
subprocess.run(["git", "-C", "Efficient-SAM3-Finetuning", "fetch", "--all"], check=True)
subprocess.run(["git", "-C", "Efficient-SAM3-Finetuning", "checkout", BRANCH], check=True)
os.chdir("Efficient-SAM3-Finetuning")
subprocess.run(["git", "log", "-1", "--oneline"], check=True)

In [ ]:
# Cell 2: Install runtime + GPU + test deps. Auto-restarts the kernel
# if a numpy major-version mismatch is detected after install.
#
# Why all these pins on ONE pip-install line:
#
# Colab ships numpy 2.x, scipy 1.16.x, and transformers 5.0.0; sam3 pins
# `numpy>=1.26,<2`. If we let pip's resolver loose with just `numpy>=1.26`
# from our project, it tries numpy 2.x first (latest matching), hits sam3's
# `<2` cap, and cascades: it backtracks through every transformers version
# (5.8.1 -> 5.0.0), then every scipy (1.17.1 -> 1.11.1-sdist), and the scipy
# sdist fails to build from source -> `metadata-generation-failed`.
#
# Splitting these into a separate pre-install does NOT work: pip re-resolves
# from scratch on the next `pip install` and re-considers upgrading numpy
# even though 1.26.4 is on disk. The pins must be on the SAME install
# command so pip honors them during the single resolution pass.
#
# Re-evaluate these pins if sam3 ever relaxes its numpy bound. The
# transformers pin is the version Colab preinstalls; bumping it is fine as
# long as the bumped version is also in the kernel's site-packages.
#
# torchao>=0.16.0 is pinned because peft 0.19+ lazily checks the installed
# torchao version on every LoRA-eligible nn.Linear dispatch
# (peft.tuners.lora.torchao calls peft.import_utils.is_torchao_available,
# which RAISES ImportError if torchao is installed AND `< 0.16.0`). Colab
# base images preinstall torchao 0.10.0, so apply_lora and apply_qlora hit
# this ImportError before any of our code runs. We do NOT use torchao
# ourselves (QLoRA uses bitsandbytes); the pin only upgrades the
# preinstalled package past peft's gate. Drop this pin once Colab ships
# torchao >= 0.16.0 by default.
#
# Why this cell auto-restarts the kernel:
#
# Modern Colab base images preload numpy 2.x into the kernel at startup
# (via matplotlib pylab autocompletion, IPython completer hooks, or
# similar). sam3 pins `numpy<2`, so this install must downgrade to
# 1.26.4 on disk -- but the kernel's in-memory numpy is still 2.x, and
# you cannot unimport a C-extension module from a running kernel. Colab
# prints "WARNING: The following packages were previously imported in
# this runtime: [numpy]. You must restart the runtime ..."
#
# We auto-kill the kernel after install when this mismatch is detected.
# Colab will show a "Your session crashed" notification and offer Run
# all. Click it -- on the re-run, the kernel starts with numpy 1.26.4
# already on disk (pip is a no-op), no mismatch, no restart. The fix is
# upstream: sam3 needs to relax its numpy pin (tracked separately).
%pip install -e ".[qlora,dev,tensorboard]" \
    "numpy==1.26.4" "scipy==1.13.1" "transformers==5.0.0" \
    "huggingface_hub>=1.15" \
    "torchao>=0.16.0"
!python -c "import esam3; print(\"esam3 OK:\", esam3.__file__)"

import sys

if "numpy" in sys.modules:
    import importlib.metadata

    _in_mem = sys.modules["numpy"].__version__
    _on_disk = importlib.metadata.version("numpy")
    if _in_mem != _on_disk:
        print()
        print(f"NUMPY DRIFT: kernel has {_in_mem} but disk has {_on_disk}.")
        print("Auto-restarting kernel. When Colab reconnects (5-10s), click")
        print("Runtime -> Run all to continue. Install is a no-op on re-run.")
        print()
        import os

        os.kill(os.getpid(), 9)


In [ ]:
# Cell 3: Environment guard. Runs AFTER the install so torch imports a numpy
# that already matches sam3's pin — fails fast if anything drifted.
import sys

import numpy
import torch

assert "google.colab" in sys.modules, "This notebook is intended for Google Colab."
assert numpy.__version__.startswith("1.26."), (
    f"numpy {numpy.__version__} != 1.26.x. Cell 2 install drifted; "
    "Runtime → Restart runtime, then Run all."
)
assert torch.cuda.is_available(), (
    "No CUDA device. Runtime → Change runtime type → T4 GPU or better."
)
cc = torch.cuda.get_device_capability()
assert cc >= (7, 5), (
    f"CUDA compute capability {cc} is < 7.5. bitsandbytes 4-bit requires Turing or later."
)
print(f"GPU OK: {torch.cuda.get_device_name(0)} (CC {cc[0]}.{cc[1]})")
print(f"numpy: {numpy.__version__}")

In [ ]:
# Cell 4: HF auth (token in Colab Secrets).
import os

from google.colab import userdata

token = userdata.get("HF_TOKEN")
assert token, "HF_TOKEN missing from Colab Secrets. See the prereqs cell."
os.environ["HF_TOKEN"] = token
os.environ["HUGGING_FACE_HUB_TOKEN"] = token  # huggingface-cli reads this name too

In [ ]:
# Cell 5: Download the SAM 3.1 checkpoint (gated; HF_TOKEN required).
# `huggingface-cli` is deprecated; the new CLI is `hf` (huggingface_hub >= 1.13).
!hf download facebook/sam3.1 --local-dir models/sam3.1

## Test cells — press play for one test

Each cell pulls the latest branch tip, runs one test (or the inspection tier as
one batch), and runs the GPU cleanup trailer regardless of pass/fail. The
inspection tier is cheap (no training) and a good first check after each push;
the training tiers are expensive and best run individually so a mid-run failure
doesn't take 40 minutes of others down with it.


In [ ]:
%%bash
# Inspection tier — 9 forward-only tests. ~5 min wall on T4.
# Validates load-time / config / structural contracts; cheapest gate.
set -e
git pull --ff-only
set +e
python -m pytest tests/integration/ \
    -v --tb=long --no-cov -m 'gpu_inspection'
TEST_EXIT=$?
echo ""
echo "=== GPU CLEANUP ==="
ORPHANS=$(nvidia-smi --query-compute-apps=pid --format=csv,noheader | tr -d ' ')
if [ -n "$ORPHANS" ]; then
    echo "Killing GPU processes still holding memory: $ORPHANS"
    echo "$ORPHANS" | xargs -r kill -9 2>/dev/null
    sleep 1
fi
nvidia-smi
echo ""
echo "Test exit code: $TEST_EXIT (0 = pass)"


In [ ]:
%%bash
# QLoRA fast smoke — 3 training steps, no final eval. ~3 min wall on T4.
# Iteration aid; same code path as the full overfit but with weaker
# assertions (finite-value only). Run after each push during dtype/grad
# debug cycles. Not a release-tier gate on its own.
set -e
git pull --ff-only
set +e
python -m pytest tests/gpu/test_real_train_qlora.py::test_qlora_smoke_fast \
    -v --tb=long --no-cov -m 'gpu'
TEST_EXIT=$?
echo ""
echo "=== GPU CLEANUP ==="
ORPHANS=$(nvidia-smi --query-compute-apps=pid --format=csv,noheader | tr -d ' ')
if [ -n "$ORPHANS" ]; then
    echo "Killing GPU processes still holding memory: $ORPHANS"
    echo "$ORPHANS" | xargs -r kill -9 2>/dev/null
    sleep 1
fi
nvidia-smi
echo ""
echo "Test exit code: $TEST_EXIT (0 = pass)"


In [ ]:
%%bash
# LoRA release-tier overfit — 50 training steps + final Evaluator.
# ~14 min wall on T4. Asserts loss drop ≤ 0.70 of start, peak VRAM ≤ 14 GB.
set -e
git pull --ff-only
set +e
python -m pytest tests/gpu/test_real_train_overfits.py \
    -v --tb=long --no-cov -m 'gpu'
TEST_EXIT=$?
echo ""
echo "=== GPU CLEANUP ==="
ORPHANS=$(nvidia-smi --query-compute-apps=pid --format=csv,noheader | tr -d ' ')
if [ -n "$ORPHANS" ]; then
    echo "Killing GPU processes still holding memory: $ORPHANS"
    echo "$ORPHANS" | xargs -r kill -9 2>/dev/null
    sleep 1
fi
nvidia-smi
echo ""
echo "Test exit code: $TEST_EXIT (0 = pass)"


In [ ]:
%%bash
# QLoRA release-tier overfit — 50 training steps + final Evaluator.
# ~14 min wall on T4. Asserts loss drop ≤ 0.75 of start, peak VRAM ≤ 10 GB.
# Strictly less weight memory than LoRA (4-bit base); should fit headroom.
set -e
git pull --ff-only
set +e
python -m pytest tests/gpu/test_real_train_qlora.py::test_qlora_overfits_in_50_steps \
    -v --tb=long --no-cov -m 'gpu'
TEST_EXIT=$?
echo ""
echo "=== GPU CLEANUP ==="
ORPHANS=$(nvidia-smi --query-compute-apps=pid --format=csv,noheader | tr -d ' ')
if [ -n "$ORPHANS" ]; then
    echo "Killing GPU processes still holding memory: $ORPHANS"
    echo "$ORPHANS" | xargs -r kill -9 2>/dev/null
    sleep 1
fi
nvidia-smi
echo ""
echo "Test exit code: $TEST_EXIT (0 = pass)"


In [ ]:
%%bash
# End-to-end CLI smoke — exercises `esam3 run` plus disk-artifact assertions
# (run dir layout, adapter save, metrics.json). ~14 min wall on T4.
# Same forward+backward machinery as the overfit tests; failures here are
# typically artifact-path or CLI-seam issues.
set -e
git pull --ff-only
set +e
python -m pytest tests/gpu/test_run_end_to_end_gpu.py \
    -v --tb=long --no-cov -m 'gpu'
TEST_EXIT=$?
echo ""
echo "=== GPU CLEANUP ==="
ORPHANS=$(nvidia-smi --query-compute-apps=pid --format=csv,noheader | tr -d ' ')
if [ -n "$ORPHANS" ]; then
    echo "Killing GPU processes still holding memory: $ORPHANS"
    echo "$ORPHANS" | xargs -r kill -9 2>/dev/null
    sleep 1
fi
nvidia-smi
echo ""
echo "Test exit code: $TEST_EXIT (0 = pass)"


## Reading test results

Scroll to the bottom of any test cell's output and read pytest's final summary
line (e.g. `========= 4 passed in 87.3s =========`). That line is the
pass/fail signal. The cleanup trailer below it confirms GPU state for the
next cell.

If a cell fails:
1. The cleanup trailer still ran — GPU memory is reclaimed; you don't need
   to restart the runtime.
2. Read the traceback above the summary line.
3. Push a fix locally, re-press the same cell. The `git pull --ff-only` at
   the top picks up the new commit.
